<a href="https://colab.research.google.com/github/wooihaw/k-youth-2603/blob/main/Lab_3/Lab_3b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 3b
#### The Steel Industry Energy Consumption dataset provides 35,040 real-world observations collected from a steel manufacturing facility in South Korea to support research in smart industry energy analytics. It contains detailed measurements of electricity usage, power factor, CO₂ emissions, temporal attributes, and operational load categories. The dataset captures both continuous and categorical features, offering a foundation for modelling energy consumption patterns and understanding the operational behaviour of industrial equipment across different time periods. Its structure makes it suitable for regression, particularly for studies on energy efficiency, load forecasting, and industrial process optimisation in modern manufacturing environments.

In [1]:
# Initialization
%matplotlib inline
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
# Load the libraries
import pandas as pd
from sklearn.model_selection import train_test_split as split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [3]:
# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/wooihaw/datasets/main/steel_industry_energy_dataset.csv')

In [5]:
# To do: Print top 5 rows of the dataset
df.head()

,date,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,1/1/2018 0:15,3.17,0.0,73.21,100.0,900.0,Weekday,Monday,Light_Load
1,1/1/2018 0:30,4.00,0.0,66.77,100.0,1800.0,Weekday,Monday,Light_Load
2,1/1/2018 0:45,3.24,0.0,70.28,100.0,2700.0,Weekday,Monday,Light_Load
3,1/1/2018 1:00,3.31,0.0,68.09,100.0,3600.0,Weekday,Monday,Light_Load
4,1/1/2018 1:15,3.82,0.0,64.72,100.0,4500.0,Weekday,Monday,Light_Load


In [6]:
# To do: Print the descriptive statistics
df.describe()

,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM
count,35040.000000,35029.000000,35040.000000,35040.000000,35031.000000
mean,27.386892,0.011527,80.578056,84.367870,42749.961463
std,33.444380,0.016152,18.921322,30.456535,24943.736035
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.200000,0.000000,63.320000,99.700000,20700.000000
50%,4.570000,0.000000,87.960000,100.000000,42300.000000
75%,51.237500,0.020000,99.022500,100.000000,64800.000000
max,157.180000,0.070000,100.000000,100.000000,85500.000000


In [7]:
# To do: Check for missing values
df.isna().sum()

date                             0
Usage_kWh                        0
CO2(tCO2)                       11
Lagging_Current_Power_Factor     0
Leading_Current_Power_Factor     0
NSM                              9
WeekStatus                       0
Day_of_week                      0
Load_Type                        0
dtype: int64

In [8]:
# If there are missing values, do the following:
# i. Impute them using statistics
# ii. Check whether there are any more missing values
df['CO2(tCO2)'] = df['CO2(tCO2)'].fillna(df['CO2(tCO2)'].mean())
df['NSM'] = df['NSM'].fillna(df['NSM'].median())
df.isna().sum()

date                            0
Usage_kWh                       0
CO2(tCO2)                       0
Lagging_Current_Power_Factor    0
Leading_Current_Power_Factor    0
NSM                             0
WeekStatus                      0
Day_of_week                     0
Load_Type                       0
dtype: int64

In [9]:
# To do: Check for unique value in the 'Load_Type' column
df['Load_Type'].unique()

array(['Light_Load', 'Medium_Load', 'Maximum_Load'], dtype=object)

In [10]:
# To do:
# i. Define the mapping for ordinal encoding of the 'Load_Type' column
# ii. Peform ordinal encoding of the 'Load_Type' column
# iii. Print the top 5 rows of the resulting dataframe
load_mapping = {'Light_Load':1, 'Medium_Load':2, 'Maximum_Load':3}
df['Load_Type'] = df['Load_Type'].map(load_mapping)
df.head()

,date,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,1/1/2018 0:15,3.17,0.0,73.21,100.0,900.0,Weekday,Monday,1
1,1/1/2018 0:30,4.00,0.0,66.77,100.0,1800.0,Weekday,Monday,1
2,1/1/2018 0:45,3.24,0.0,70.28,100.0,2700.0,Weekday,Monday,1
3,1/1/2018 1:00,3.31,0.0,68.09,100.0,3600.0,Weekday,Monday,1
4,1/1/2018 1:15,3.82,0.0,64.72,100.0,4500.0,Weekday,Monday,1


In [11]:
# To do:
# i. Perform one-hot encoding of the 'WeekStatus' and 'Day_of_week' columns
# ii. Print the top 5 rows of the resulting dataframe
df2 = pd.get_dummies(df, columns=['WeekStatus', 'Day_of_week'])
df2.head()

,date,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,Load_Type,WeekStatus_Weekday,WeekStatus_Weekend,Day_of_week_Friday,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday
0,1/1/2018 0:15,3.17,0.0,73.21,100.0,900.0,1,True,False,False,True,False,False,False,False,False
1,1/1/2018 0:30,4.00,0.0,66.77,100.0,1800.0,1,True,False,False,True,False,False,False,False,False
2,1/1/2018 0:45,3.24,0.0,70.28,100.0,2700.0,1,True,False,False,True,False,False,False,False,False
3,1/1/2018 1:00,3.31,0.0,68.09,100.0,3600.0,1,True,False,False,True,False,False,False,False,False
4,1/1/2018 1:15,3.82,0.0,64.72,100.0,4500.0,1,True,False,False,True,False,False,False,False,False


In [12]:
# To do:
# i. Separate features and target into X and y, respectively. Drop any irrelevant features if necessary.
# ii. Split to training and testing sets with 20% of the data samples for testing
X = df2.drop(columns=['Usage_kWh', 'date'])
y = df2['Usage_kWh']
X_train, X_test, y_train, y_test = split(X, y, test_size=0.2, random_state=42)

In [15]:
# To do: Use spot-checking technique with 5-fold cross validation to quickly evaluate the performance of Linear Regression, kNN Regression, Decision Tree Regression, Random Forest Regression, Gradient Boosting Tree Regression and Multilayer Perceptron Regression.
models = {}
models['knn'] = KNeighborsRegressor()
models['lnr'] = LinearRegression()
models['dtr'] = DecisionTreeRegressor()
models['rfr'] = RandomForestRegressor()
models['gbr'] = GradientBoostingRegressor()
models['mlr'] = MLPRegressor()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for n in models:
    scores = cross_val_score(models[n], X_train, y_train, cv=kf, n_jobs=-1)
    print(f'{n}: {scores.mean():.3f} +/- {scores.std():.3f}')

knn: 0.794 +/- 0.005
lnr: 0.975 +/- 0.002
dtr: 0.979 +/- 0.001
rfr: 0.988 +/- 0.000
gbr: 0.982 +/- 0.002
mlr: 0.620 +/- 0.018


In [16]:
# To do: Apply MinMaxScaling to the features
scl = MinMaxScaler()
Xs_train = scl.fit_transform(X_train)
Xs_test = scl.transform(X_test)


In [17]:
# To do: Use spot-checking technique with 5-fold cross validation to quickly evaluate the performance of Linear Regression, kNN Regression, Decision Tree Regression, Random Forest Regression, Gradient Boosting Tree Regression and Multilayer Perceptron Regression on the scaled features
for n in models:
    scores = cross_val_score(models[n], Xs_train, y_train, cv=kf, n_jobs=-1)
    print(f'{n}: {scores.mean():.3f} +/- {scores.std():.3f}')

knn: 0.985 +/- 0.000
lnr: 0.975 +/- 0.002
dtr: 0.979 +/- 0.001
rfr: 0.988 +/- 0.000
gbr: 0.982 +/- 0.002
mlr: 0.976 +/- 0.002


In [18]:
# To do:
# i. Train the best algorithm using the entire training set with scaled features
# ii. Evaluate the best model using the testing set
best_model = RandomForestRegressor(random_state=42).fit(Xs_train, y_train)
print(f'Best model R2-Score: {best_model.score(Xs_test, y_test):.3f}')

Best model R2-Score: 0.989


In [19]:
# To do: Evaluate the best model in terms of MAE and RMSE
y_pred = best_model.predict(Xs_test)
print(mean_absolute_error(y_test, y_pred))
print(root_mean_squared_error(y_test, y_pred))

2.0542551728296905
3.5974637963096976


In [20]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

xgbr = XGBRegressor().fit(Xs_train, y_train)
print(f'xgbr R2-Score: {xgbr.score(Xs_test, y_test):.3f}')

xgbr R2-Score: 0.989


In [21]:
lgbmr = LGBMRegressor().fit(Xs_train, y_train)
print(f'lgbmr R2-Score: {lgbmr.score(Xs_test, y_test):.3f}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000441 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 28032, number of used features: 14
[LightGBM] [Info] Start training from score 27.286074
lgbmr R2-Score: 0.989
